# Jiu-jitsu model explorer

Explore saved experiments, inspect validation curves and class metrics, and run
checkpoint predictions on random frames with supplied-pose overlays.
The notebook reads `trainings/` directly on the DGX. It does not train models.

**Before running:** select the **JiujitsuCompVision (.venv)** kernel.
If it is not installed, run these commands once in a JupyterLab terminal,
from the project root:

```bash
.venv/bin/python -m pip install -r install/requirements-notebook.txt
.venv/bin/python -m ipykernel install --user --name jiujitsu-compvision --display-name "JiujitsuCompVision (.venv)"
```

Run the setup cells from top to bottom. Loading a checkpoint and making
predictions require explicit button clicks; **Run All** only prepares the browsers.

In [ ]:
from pathlib import Path
import os
import sys

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/Modules/models.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the JiujitsuCompVision checkout.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Change these paths if data or experiments live outside this checkout.
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "Data"))
TRAININGS_DIR = Path(os.environ.get("TRAININGS_DIR", ROOT / "trainings"))
ANNOTATIONS = DATA_DIR / "annotations.json"
SPLIT_FILE = Path(os.environ.get("SPLIT_FILE", DATA_DIR / "splits/multiview_sections.json"))
IMAGES_DIR = DATA_DIR / "images"
DEVICE = os.environ.get("JIUJITSU_NOTEBOOK_DEVICE", "cpu")  # e.g. "cuda:4"
SEED = 42
POSE_INITIAL_EPOCHS = 20

print("Project:", ROOT)
print("Kernel Python:", sys.executable)
print("Metrics:", TRAININGS_DIR)
print("Split:", SPLIT_FILE)

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.Eval.exploration import ExplorationData, ModelExplorer
from src.Eval.exploration_plots import (
    catalog_runs, load_run, plot_training_curves, plot_run_comparison,
    plot_class_recall, plot_confusion_matrix,
)
from src.Eval.exploration_ui import RunBrowser, PredictionBrowser, plot_predictions

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
for index in range(torch.cuda.device_count()):
    print(f"cuda:{index}: {torch.cuda.get_device_name(index)}")
print("Inference default:", DEVICE, "(editable in the prediction browser)")

## Training curves and class results

Choose a run and click **Show**. Use **Refresh runs** to pick up new epochs.
The catalog includes unfinished runs; its status describes the log, not process state.

- **Training curves:** train/validation loss, accuracy, and validation macro F1.
  Pose runs separate the first 20 epochs from later epochs; change **Pose initial**
  or set it to `0` for the full trajectory.
- **Class recall:** correct examples per class, including held-out view and unseen moment.
- **Confusion matrix:** true classes on rows, predicted classes on columns.
  **Cohort** selects the evaluation group.
- **Configuration:** saved hyperparameters and architecture settings.

Class figures use the best pooled-validation-F1 epoch by default. Enter `last`
or an epoch number to inspect another logged epoch. These are validation results.

In [ ]:
# Reuse the browser so prediction controls keep the same selected-run reference.
if "browser" in globals() and browser.trainings_dir == TRAININGS_DIR:
    browser.refresh()
else:
    browser = RunBrowser(TRAININGS_DIR)
    if "predictor" in globals():
        predictor.close()
        predictor.run_browser = browser
browser.initial_epochs.value = POSE_INITIAL_EPOCHS
display(browser.widget)

## Compare selected runs

Optional: paste run-directory names below, or use `browser.run_name.value`.
The comparison checks annotation and split hashes. Image and pose models may
still use different eligible examples. Choose `held_out_view_macro_f1` or
`unseen_moment_macro_f1` to compare those cohorts.

In [ ]:
COMPARE_RUNS = []  # e.g. [browser.run_name.value, "another-run-directory"]
COMPARE_METRIC = "val_macro_f1"

if COMPARE_RUNS:
    compared_runs = [load_run(TRAININGS_DIR, name) for name in COMPARE_RUNS]
    comparison = plot_run_comparison(compared_runs, metric=COMPARE_METRIC)
    display(comparison)
    plt.close(comparison)
    for number, name in enumerate(comparison.run_names, 1):
        print(f"{number}: {name}")
else:
    print("Add run names to COMPARE_RUNS, then re-run this cell.")

## Dataset and checkpoint predictions

The next cell loads annotations and verifies the shared split. If a historical
checkpoint used another split version, set `SPLIT_FILE` to that manifest and
re-run setup/data cells. A mismatch is reported rather than silently evaluating
on different data.

Validation is the default for exploration. Test is an explicit option; keep
final-test examples separate from choices about architecture and hyperparameters.

In [ ]:
for required in (ANNOTATIONS, SPLIT_FILE):
    if not required.is_file():
        raise FileNotFoundError(f"Required dataset file is missing: {required}")

if "predictor" in globals():
    predictor.close()
data = ExplorationData(ANNOTATIONS, SPLIT_FILE, IMAGES_DIR)
if "predictor" in globals():
    predictor.set_data(data)
print(f"{len(data.records):,} annotated frames, {len(data.sources)} sources.")
for partition in ("train", "val", "test"):
    images = len(data.select("image", split=partition))
    poses = len(data.select("pose", split=partition))
    print(f"{partition}: {images:,} image examples; {poses:,} eligible two-pose examples")
print("Annotation hash:", data.annotation_sha256)
print("Split hash:", data.split_sha256)

### Load and sample

1. Select the run in the metrics browser above.
2. Choose `best` or `last`, enter `cpu` or a visible device such as `cuda:4`,
   and click **Load selected run**.
3. Choose filters and click **New random samples**.

**Incorrect**, **Correct**, and **Lowest confidence** search a bounded random
pool, then display a few results. Increase **Search pool** to look further.
An empty result means no match in that pool, not that the model never makes
that kind of prediction. **Repeatable sample** uses the seed; otherwise each
click draws new examples.

**Predict this image** accepts an exact image ID under the current filters.
**Show model input** reveals the actual input tensor of a displayed example.
The overlay threshold only changes drawing. Inference always uses the saved
preprocessing, no augmentation, and eval mode.

Pose models exclude missing-athlete and other pose-ineligible examples.
The colored skeletons are supplied annotations, not poses predicted from the
image. Softmax values are model scores, not calibrated probabilities.

In [ ]:
if "predictor" in globals():
    predictor.close()
predictor = PredictionBrowser(data, browser, device=DEVICE)
predictor.seed.value = SEED
display(predictor.widget)

## Inspect the loaded architecture and exact features

After loading a checkpoint, re-run this cell to inspect its architecture and
saved model configuration. For a displayed pose sample, **Show model input**
also stores the exact tensor as `predictor.last_input_tensor`, which can be
reshaped to `(2, 17, 3)`.

In [ ]:
if predictor.session is None:
    print("Load a checkpoint in the prediction browser first.")
else:
    print("Checkpoint:", predictor.session.checkpoint_path)
    print("Epoch:", predictor.session.checkpoint_epoch)
    print("Parameter counts:", predictor.session.parameter_counts)
    display(predictor.session.config["model_config"])
    print(predictor.session.model)

## Optional: compare predictions on the same frames

Enter run names below. This uses a shared, repeatable sample with both athlete
poses so pose and image models see the same IDs. Models load sequentially and
release memory after inference. Every checkpoint must match the configured
dataset and split. This small sample is for inspection, not a model ranking.

In [ ]:
PREDICTION_RUNS = []  # e.g. ["pose-run-directory", "image-run-directory"]
COMPARE_COUNT = 3

if PREDICTION_RUNS:
    predictor.close()
    shared_records = data.sample("pose", split="val", count=COMPARE_COUNT, seed=SEED)
    shared_predictions = {}
    for name in PREDICTION_RUNS:
        model = ModelExplorer.from_run(TRAININGS_DIR / name, data, checkpoint="best", device=DEVICE)
        try:
            results = model.predict(shared_records, top_k=3, batch_size=16)
            shared_predictions[name] = results
            print(name)
            figure = plot_predictions(data, shared_records, results, overlay=True)
            display(figure)
            plt.close(figure)
        finally:
            model.close()
else:
    print("Add run names to PREDICTION_RUNS to compare the same validation frames.")

## Continue in Python

The browsers expose their state for further exploration:

```python
browser.catalog                 # latest run summaries
browser.run                     # most recently plotted Run
predictor.records               # currently displayed canonical annotations
predictor.predictions           # classes, softmax scores, source/frame/cohort
predictor.last_input_tensor     # available after Show model input
data.describe("0300001")        # metadata for an existing image ID
```

Use `data.select(...)` for a chronological slice, sort by `record["frame"]`,
and pass it to `predictor.session.predict(...)` to inspect a sequence.
Keep the slice bounded when experimenting with image models.

When finished, call `predictor.close()` or shut down the kernel to release its
model memory. Clear cell outputs before committing an executed notebook.